In [1]:
#libraries
import pandas as pd
from datetime import datetime, timedelta
import sys
sys.path.insert(0, "C:/Soft/repos/cwms-python")
import cwms
import numpy as np
import time

ModuleNotFoundError: No module named 'cwms'

### Initialize system Load Location IDs

In [2]:
office_id = 'LRL'
start_date = pd.to_datetime("3/1/2025").tz_localize('UTC')
end_date = pd.to_datetime("6/13/2025").tz_localize('UTC')
office_id_lower = office_id.lower() 
apiRoot_src = f"https://wm.{office_id_lower}.ds.usace.army.mil:8243/{office_id_lower}-data/"
location_id_file = 'base_locations_to_grab.csv'
api = cwms.api.init_session(api_root=apiRoot_src)
grab_locs = pd.read_csv(location_id_file)

### Grab all locations and the get information for subset

In [6]:
location = cwms.get_locations_catalog(office_id=office_id).df
grab_locs_office = grab_locs[grab_locs['Office']==office_id]
pattern = "|".join(grab_locs_office['Base_Location'])
public_locations = location[location['name'].str.contains(pattern)]
public_locations.to_csv(f'data/{office_id}_locations_data.csv', index=False)

### Collect All Timeseries ID assigned to the Locations

In [ ]:
final_tsids =[]
for loc in public_locations['name']:
    loc_repex = loc+'.*'
    cat_ts = cwms.get_timeseries_catalog(office_id=office_id,like=loc_repex,include_extents=True,timeseries_group_like=None).df
    for index, ts in cat_ts.iterrows():

        extents = pd.DataFrame(ts['extents'])
        if 'latest-time' in extents.columns:
            if len(extents) == 1 and (pd.to_datetime(extents['latest-time'].iloc[0]) > start_date):

                final_tsids.append(ts['name'])
            if len(extents) > 1:

                extents['version-time'] = pd.to_datetime(extents['version-time'])
                extents = extents[extents['version-time'] > start_date]
                if len(extents) > 0:
                    num = min([len(extents),5])
                    extents_sorted = extents.sort_values(by='version-time',ascending=False)
                    extents_tosave = extents_sorted.iloc[0:num]
                    for version in extents_tosave['version-time']:
                        final_tsids.append(f'{ts["name"]}:{version}')
ts_ids = pd.DataFrame({'office':office_id,'ts_id':final_tsids})              
ts_ids = ts_ids.drop_duplicates()
ts_ids.to_csv(f'data/{office_id}_timeseries_ids.csv')

### Grab Timeseries Values for All Timeseries IDS

In [10]:
ts_ids = pd.read_csv(f'data/{office_id}_timeseries_ids.csv')

In [11]:
start_time = time.perf_counter()
multi_ts_melt = cwms.get_multi_timeseries_df(ts_ids=ts_ids['ts_id'],office_id=office_id,melted=True,begin=start_date,end=end_date)
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time} seconds")
multi_ts_melt.to_parquet(f'data/{office_id}_timeseries_values_melted.parquet')

Elapsed time: 59.33675069999998 seconds


In [2]:
multi_ts_melt = pd.read_parquet('data/MVP_timeseries_values_melted.parquet')
apiRoot_dev = "https://water.dev.cwbi.us/cwms-data/"

In [3]:
from getpass import getpass
apiKey = getpass()
apiKey_dev = "apikey " + apiKey

 ········


In [4]:
api = cwms.api.init_session(api_root=apiRoot_dev, api_key=apiKey_dev)

In [5]:
cwms.store_multi_timeseries_df(ts_data=multi_ts_melt,office_id='MVP')

In [10]:
import time
start_time = time.perf_counter()
multi_ts_melt_dev = cwms.get_multi_timeseries_df(ts_ids=final_tsids,office_id='MVP',melted=True,begin=start_date,end=end_date)
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time} seconds")

NameError: name 'start_date' is not defined

In [52]:
unique_tsids = (multi_ts_melt['ts_id'].astype(str) + ':' + multi_ts_melt['version_date'].astype(str)).unique()

In [53]:
unique_tsids

array(['LockDam_05-TainterGate23.Flow.Inst.15Minutes.0.comp:NaT',
       'LockDam_05-TainterGate23.Opening.Inst.15Minutes.0.CEMVP-ProjectEntry:NaT',
       'LockDam_05-TainterGate23.Opening.Inst.~15Minutes.0.CEMVP-ProjectEntry:NaT',
       ..., 'LockDam_02.Temp-Water.Inst.15Minutes.0.merged:NaT',
       'LockDam_02.Temp-Water.Inst.~1Day.0.Raw-NWS-IEM:NaT',
       'LockDam_02.Volt.Inst.1Hour.0.CEMVP-GOES-Raw:NaT'], dtype=object)

In [55]:
data['units'].iloc[0]

'cfs'

In [ ]:
data = multi_ts_melt.query('ts_id' == 'LockDam_05-TainterGate23.Flow.Inst.15Minutes.0.comp' and 'version_date' < 9')

In [45]:
cols = ["ts_id", "units"]
if "version_date" in multi_ts.columns:
    cols.append("version_date")
    multi_ts["version_date"] = multi_ts["version_date"].dt.strftime(
        "%Y-%m-%d %H:%M:%S%z"
    )
    multi_ts["version_date"] = (
        multi_ts["version_date"].str[:-2] + ":" + multi_ts["version_date"].str[-2:]
    )
    multi_ts.fillna({"version_date": ""}, inplace=True)
multi_ts_unmelt = multi_ts.pivot(index="date-time", columns=cols, values="value")

ValueError: Index contains duplicate entries, cannot reshape

In [13]:


multi_ts_melt

,date-time,value,quality-code,ts_id,units
0,2025-03-01 00:00:00+00:00,0.0,0,LockDam_05-TainterGate23.Opening.Inst.15Minute...,ft
1,2025-03-01 00:15:00+00:00,0.0,0,LockDam_05-TainterGate23.Opening.Inst.15Minute...,ft
2,2025-03-01 00:30:00+00:00,0.0,0,LockDam_05-TainterGate23.Opening.Inst.15Minute...,ft
3,2025-03-01 00:45:00+00:00,0.0,0,LockDam_05-TainterGate23.Opening.Inst.15Minute...,ft
4,2025-03-01 01:00:00+00:00,0.0,0,LockDam_05-TainterGate23.Opening.Inst.15Minute...,ft
...,...,...,...,...,...
4129350,2025-06-06 10:30:00+00:00,69.4,0,LockDam_02.Temp-Water.Inst.~1Day.0.Raw-NWS-IEM,F
4129351,2025-06-07 10:00:00+00:00,69.3,0,LockDam_02.Temp-Water.Inst.~1Day.0.Raw-NWS-IEM,F
4129352,2025-06-09 10:00:00+00:00,67.0,0,LockDam_02.Temp-Water.Inst.~1Day.0.Raw-NWS-IEM,F
4129353,2025-06-11 10:00:00+00:00,68.8,0,LockDam_02.Temp-Water.Inst.~1Day.0.Raw-NWS-IEM,F
